In [380]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, classification_report, multilabel_confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier

Adding the area of the regions to the dataframe.

First, we'll create a dictionary with area codes and areas, then we iterate over a list of area codes from the accidents, create a new array based on them and add it to the dataframe.

In [381]:
accidents = pd.read_csv("koeti_riista_tau_001_en.csv", encoding='iso-8859-1')

regions = pd.read_csv('maakunnat_pinta_alat.csv')

areas = list(regions.iloc[:, 5])
area_codes = list(regions.iloc[:, 2])
areas_dict = dict(zip(area_codes, areas))

accident_areas = list(accidents.iloc[:, 8])

areas_arr = []

for accident_area in accident_areas:
    areas_arr.append(int(areas_dict[accident_area]))

accidents['regionArea'] = areas_arr

Deriving time of day in seconds from eventTime.

First, we'll assign the values from the eventTime column into an array, then we'll iterate over the arrays strings and convert them to seconds, finally we append them to an array and add it to the dataframe.

In [ ]:
seconds_arr = []

times = accidents.iloc[:, [1]].values

# '2017-12-10T16:55:00.000+02:00'

for time in times:
    times_of_day = time[0].split('T')[1]
    tz = times_of_day.split('+')[1].split(':')[0][1]
    time_of_day = times_of_day.split('+')[0].split('.')[0].split(':')
    seconds = (int(time_of_day[0]) + int(tz)) * 3600 + int(time_of_day[1]) * 60 + int(time_of_day[2])
    seconds_arr.append(seconds)

accidents['timeOfDay'] = seconds_arr

Converting species names to one-hot values

With pandas get_dummies function, we'll first generate a dataframe with one-hot values for every species. Then, for the merge function to work, we add an id column to it and merge with accidents dataframe.

In [383]:
species_oh = pd.get_dummies(accidents['wildlifeSpeciesName'], dtype=int)

species_oh['id'] = range(len(species_oh))

accidents = accidents.merge(species_oh, on='id')

Next we'll drop unnecessary columns and then rows with null values.

In [384]:
print(len(accidents))

print(len(accidents.dropna()))

accidents = accidents.drop(columns=['eventTime', 'municipalityName', 'regionName', 'roadTypeShort', 'roadMaintenanceName', 'wildlifeSpecies', 'wildlifeSpeciesName', 'roadNumber', 'roadMaintenance'])

accidents = accidents.dropna()

print(len(accidents))

63941
54511
63937


In [385]:
accidents.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63937 entries, 0 to 63940
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               63937 non-null  int64  
 1   year             63937 non-null  int64  
 2   month            63937 non-null  int64  
 3   x                63937 non-null  float64
 4   y                63937 non-null  float64
 5   municipality     63937 non-null  int64  
 6   region           63937 non-null  int64  
 7   roadType         63937 non-null  int64  
 8   regionArea       63937 non-null  int64  
 9   timeOfDay        63937 non-null  int64  
 10  Hirvi            63937 non-null  int64  
 11  Kuusipeura       63937 non-null  int64  
 12  Metsäkauris      63937 non-null  int64  
 13  Metsäpeura       63937 non-null  int64  
 14  Valkohäntäpeura  63937 non-null  int64  
 15  Villisika        63937 non-null  int64  
dtypes: float64(2), int64(14)
memory usage: 8.3 MB


Data sharing

We first separate feature and label columns from the dataframe, then split and fit them to a linear regression model.

In [386]:
features = ['year', 'month', 'municipality', 'region', 'roadType', 'regionArea', 'timeOfDay', 'x', 'y']
labels = ['Hirvi', 'Kuusipeura', 'Metsäkauris', 'Metsäpeura', 'Valkohäntäpeura', 'Villisika']

X, y = accidents[features].values, accidents[labels].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print(f"Mean squared error: {mse:.2f}")
print(f"R2 score: {r2:.2f}")


Mean squared error: 0.10
R2 score: 0.01


Model training

Next, we train a decision tree using the training data and create a prediction

In [387]:
clf = DecisionTreeClassifier()

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

Model evaluation

Finally, we evaluate the results using various evaluation functions from sklearn.metrics

In [390]:
print(classification_report(y_test, y_pred))

print(multilabel_confusion_matrix(y_test, y_pred))

print()
print(accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.18      0.19      0.19      4485
           1       0.00      0.00      0.00        39
           2       0.39      0.40      0.39     11256
           3       0.03      0.03      0.03       171
           4       0.54      0.53      0.53     15933
           5       0.01      0.01      0.01        85

   micro avg       0.43      0.43      0.43     31969
   macro avg       0.19      0.19      0.19     31969
weighted avg       0.43      0.43      0.43     31969
 samples avg       0.43      0.43      0.43     31969

[[[23448  4036]
  [ 3613   872]]

 [[31872    58]
  [   39     0]]

 [[13860  6853]
  [ 6807  4449]]

 [[31646   152]
  [  166     5]]

 [[ 8961  7075]
  [ 7567  8366]]

 [[31785    99]
  [   84     1]]]

0.42832118614908193


e:\Koulu\3_Vuosi\Data-analytics\data-analytics\.venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
